# Hotel Booking Cancellation Prediction — Modeling

This notebook builds a reproducible modeling workflow for predicting `is_canceled` at booking time. We use the data-quality findings from the EDA, prevent target leakage, compare a linear baseline with tree-based models, and evaluate the final candidates with metrics that are informative for both classes.

The notebook is intentionally focused on model development. Reusable training and prediction scripts will be extracted after we agree on the final feature and model policy.

## 1. Setup and reproducibility

We import the libraries needed for data preparation, cross-validation, model fitting, and evaluation. We also fix a random seed so that the split and randomized searches can be reproduced.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

RANDOM_STATE = 42
TEST_SIZE = 0.20
CV_FOLDS = 5
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid", context="notebook")

The environment is ready. The workflow uses a stratified split because cancellation is a binary target with a meaningful, but not extreme, class imbalance. The untouched test set will be used only after model selection.

## 2. Load the source data and define the feature policy

We load `agent` and `company` as categorical identifiers, matching the EDA treatment. We then remove exact duplicate rows before splitting. This prevents identical records from crossing the train, validation, and test partitions and producing an overly optimistic estimate.

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "hotel_bookings.csv"

bookings_raw = pd.read_csv(DATA_PATH, dtype={"agent": "str", "company": "str"})
bookings = bookings_raw.drop_duplicates(keep="first").copy()

TARGET = "is_canceled"
DIRECT_LEAKAGE = ["reservation_status", "reservation_status_date"]
STRICT_TIMING_EXCLUDED = ["booking_changes", "days_in_waiting_list"]
EXCLUDED_FEATURES = [TARGET] + DIRECT_LEAKAGE + STRICT_TIMING_EXCLUDED + ["company"]

X = bookings.drop(columns=EXCLUDED_FEATURES)
y = bookings[TARGET].astype(int)

print(f"Raw rows: {len(bookings_raw):,}")
print(f"Rows after exact deduplication: {len(bookings):,}")
print(f"Removed duplicate rows: {len(bookings_raw) - len(bookings):,}")
print(f"Target cancellation rate after deduplication: {y.mean():.1%}")
print(f"Model input columns: {X.shape[1]}")

Raw rows: 119,390
Rows after exact deduplication: 87,396
Removed duplicate rows: 31,994
Target cancellation rate after deduplication: 27.5%
Model input columns: 26


The EDA retained duplicate rows for analysis and documented their impact. For this modeling experiment, we remove exact duplicate rows before splitting so identical records cannot cross the train/test boundary and inflate validation performance.

The strict booking-time policy excludes the target, fields that directly reveal the final outcome (`reservation_status` and `reservation_status_date`), and fields whose definitions depend on events after booking entry (`booking_changes` and `days_in_waiting_list`). Other source fields remain available under the project assumption that their recorded booking-time values can be used. We exclude `company` because 94.3% of its values are missing, while retaining `agent` and `country` so their missingness can be represented explicitly during preprocessing.

Exact deduplication is a modeling safeguard, not an assertion that every repeated row is an erroneous record. The EDA showed that duplicate predictor values do not conflict on the target, but retaining all copies could allow the same observation to appear in both partitions.

## 3. Train, validation, and test split

We reserve a stratified test set before any model selection. We then split the remaining data into training and validation partitions. Cross-validation and preprocessing are performed on the training partition, the validation partition is used to compare models and thresholds, and the untouched test set is used only for the final evaluation.

In [3]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_train_val,
)

categorical_features = X_train.select_dtypes(include=["str"]).columns.tolist()
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()

print(f"Train rows: {len(X_train):,}; validation rows: {len(X_val):,}; test rows: {len(X_test):,}")
print(f"Train cancellation rate: {y_train.mean():.1%}")
print(f"Validation cancellation rate: {y_val.mean():.1%}")
print(f"Test cancellation rate: {y_test.mean():.1%}")
print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

Train rows: 55,932; validation rows: 13,984; test rows: 17,480
Train cancellation rate: 27.5%
Validation cancellation rate: 27.5%
Test cancellation rate: 27.5%
Numeric features: 15
Categorical features: 11


The two stratified splits preserve the cancellation share across training, validation, and test partitions. The feature groups are inferred from the training data; `agent` is encoded categorically, while `company` was excluded before the split.

## 4. Preprocessing pipelines

We create two preprocessing variants. Logistic regression benefits from standardized numeric inputs, while tree-based models do not require scaling. Both variants impute missing values and one-hot encode categorical levels inside the estimator pipeline.

We use feature-specific missing-value policies: the four missing values in `children` are imputed with `0`, while missing categorical values are replaced with the explicit category `Missing`. No other numeric column contains missing values. The `company` column is excluded before this stage because 94.3% of its values are missing.

In [4]:
numeric_linear = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
        ("scaler", StandardScaler()),
    ]
)
numeric_tree = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="constant", fill_value=0))]
)
categorical = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor_linear = ColumnTransformer(
    transformers=[
        ("num", numeric_linear, numeric_features),
        ("cat", categorical, categorical_features),
    ]
)
preprocessor_tree = ColumnTransformer(
    transformers=[
        ("num", numeric_tree, numeric_features),
        ("cat", categorical, categorical_features),
    ]
)

The preprocessing objects are kept inside each model pipeline. This makes cross-validation honest and gives the future training script one serialized object containing both transformations and the estimator.

## 5. Evaluation helpers and a simple reference point

We report ROC AUC for ranking quality and precision, recall, F1, and accuracy at the default 0.5 threshold. The class-level report makes performance on cancellations and non-cancellations visible. Accuracy alone would hide the practical trade-off between missed cancellations and false alarms.

In [5]:
def evaluate_classifier(name, estimator, X_eval, y_eval, threshold=0.5):
    probabilities = estimator.predict_proba(X_eval)[:, 1]
    predictions = (probabilities >= threshold).astype(int)
    precision = precision_score(y_eval, predictions, zero_division=0)
    recall = recall_score(y_eval, predictions, zero_division=0)
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    metrics = {
        "model": name,
        "roc_auc": roc_auc_score(y_eval, probabilities),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": accuracy_score(y_eval, predictions),
    }
    print(f"ROC AUC: {metrics['roc_auc']:.3f}")
    print(classification_report(y_eval, predictions, digits=3, zero_division=0))
    return metrics

The majority-class baseline provides a minimum reference for accuracy, but it has no useful recall for the minority cancellation class. The trained models must improve the class-sensitive metrics rather than only match this baseline.

## 6. Logistic regression baseline

We fit a regularized logistic regression as an interpretable linear baseline. Its hyperparameter search varies only the regularization strength, keeping the first comparison small and reproducible.

In [6]:
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor_linear),
        ("model", LogisticRegression(max_iter=500, class_weight=None, random_state=RANDOM_STATE)),
    ]
)

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
logistic_search = GridSearchCV(
    logistic_pipeline,
    param_grid={"model__C": np.logspace(-2, 1, 8)},
    scoring="roc_auc",
    cv=cv,
    n_jobs=1,
    refit=True,
)
logistic_search.fit(X_train, y_train)
print(f"Best logistic ROC AUC (CV): {logistic_search.best_score_:.3f}")
print(logistic_search.best_params_)

Best logistic ROC AUC (CV): 0.862
{'model__C': np.float64(1.3894954943731375)}


The logistic search selects regularization using cross-validated ROC AUC on the training partition only. We keep the fitted best estimator for the validation evaluation below.

In [7]:
results = []
results.append(evaluate_classifier("Logistic regression", logistic_search.best_estimator_, X_val, y_val))

ROC AUC: 0.864
              precision    recall  f1-score   support

           0      0.836     0.909     0.871     10140
           1      0.688     0.530     0.598      3844

    accuracy                          0.805     13984
   macro avg      0.762     0.719     0.735     13984
weighted avg      0.795     0.805     0.796     13984



The cross-validated ROC AUC and the validation ROC AUC are compared during model selection. The test partition remains untouched until the final pipeline and threshold have been selected.

## 7. Random Forest

We next tune a Random Forest to capture nonlinear effects and interactions without requiring numeric scaling. Randomized search explores a controlled parameter space and selects the configuration with the highest cross-validated ROC AUC.

In [8]:
forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor_tree),
        ("model", RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=1,
            class_weight=None,
        )),
    ]
)

forest_search = RandomizedSearchCV(
    forest_pipeline,
    param_distributions={
        "model__n_estimators": [50, 100],
        "model__max_depth": [12, 20],
        "model__min_samples_leaf": [1, 2, 5],
        "model__max_features": ["sqrt", 0.5],
        "model__class_weight": [None, "balanced"],
    },
    n_iter=3,
    scoring="roc_auc",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=1,
    refit=True,
)
forest_search.fit(X_train, y_train)
print(f"Best Random Forest ROC AUC (CV): {forest_search.best_score_:.3f}")
print(forest_search.best_params_)

Best Random Forest ROC AUC (CV): 0.885
{'model__n_estimators': 50, 'model__min_samples_leaf': 5, 'model__max_features': 'sqrt', 'model__max_depth': 20, 'model__class_weight': 'balanced'}


The tuned forest is evaluated on the validation set and compared with the other tuned models.

In [9]:
results.append(evaluate_classifier("Random Forest", forest_search.best_estimator_, X_val, y_val))

ROC AUC: 0.884
              precision    recall  f1-score   support

           0      0.929     0.744     0.826     10140
           1      0.557     0.850     0.673      3844

    accuracy                          0.773     13984
   macro avg      0.743     0.797     0.750     13984
weighted avg      0.827     0.773     0.784     13984



The tuned forest gives us a nonlinear benchmark. We compare its cross-validated and validation ROC AUC with the other tuned models before selecting a final candidate.

## 8. XGBoost

XGBoost is a primary candidate for this tabular dataset because boosted trees can model nonlinear relationships and interactions efficiently. We use the same preprocessing and cross-validation policy as the other models.

In [ ]:
xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor_tree),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=1,
        )),
    ]
)
xgb_search = RandomizedSearchCV(
    xgb_pipeline,
    param_distributions={
        "model__n_estimators": [50, 100, 200],
        "model__max_depth": [3, 5, 7],
        "model__learning_rate": [0.03, 0.07, 0.12],
        "model__subsample": [0.7, 0.9, 1.0],
        "model__colsample_bytree": [0.7, 0.9, 1.0],
        "model__min_child_weight": [1, 3, 8],
    },
    n_iter=20,
    scoring="roc_auc",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=1,
    refit=True,
)
xgb_search.fit(X_train, y_train)
print(f"Best XGBoost ROC AUC (CV): {xgb_search.best_score_:.3f}")
print(xgb_search.best_params_)
results.append(evaluate_classifier("XGBoost", xgb_search.best_estimator_, X_val, y_val))

XGBoost is evaluated with ROC AUC during cross-validation, while the final report still includes precision, recall, F1, and accuracy at the default threshold. The threshold analysis below evaluates this trade-off using the validation partition.

## 9. Model comparison and error review

We consolidate the validation-set metrics in one table. The validation set is used to compare candidate models, while the test set remains untouched for the final evaluation.

In [ ]:
results_table = pd.DataFrame(results).sort_values("roc_auc", ascending=False).reset_index(drop=True)
display(results_table.style.format({
    "roc_auc": "{:.3f}",
    "precision": "{:.3f}",
    "recall": "{:.3f}",
    "f1": "{:.3f}",
    "accuracy": "{:.3f}",
}))

We select XGBoost as the preferred model family based on its stronger validation ROC AUC and balanced classification metrics. The full, reduced, and engineered feature sets are compared in the following sections before finalizing the workflow.

## 10. XGBoost feature importance

We inspect the feature importance values of the full XGBoost candidate to understand which transformed inputs contribute most to its predictions. Because categorical variables are one-hot encoded, the table reports the encoded category-level features as well as the original numeric variables.

In [ ]:
best_xgb_pipeline = xgb_search.best_estimator_
xgb_preprocessor = best_xgb_pipeline.named_steps["preprocessor"]
xgb_model = best_xgb_pipeline.named_steps["model"]

transformed_feature_names = xgb_preprocessor.get_feature_names_out()
feature_importance = pd.DataFrame({
    "feature": transformed_feature_names,
    "importance": xgb_model.feature_importances_,
}).sort_values("importance", ascending=False)

display(feature_importance.head(20).style.format({"importance": "{:.4f}"}))

The feature-importance table provides a model-specific interpretation of the selected XGBoost pipeline. These values describe relative contribution within this fitted model; they do not establish causality and should be interpreted alongside the leakage policy and the EDA findings.

## 11. Full versus reduced feature set

We compare the full XGBoost pipeline using all permitted features with a reduced version based on model-derived importance. We aggregate one-hot encoded importances back to their original columns and retain the smallest set of original features that accounts for at least 90% of the total importance. The feature selection uses only the training data; both versions are then evaluated on the same validation set.

In [ ]:
def original_feature_name(transformed_name, original_features):
    for original_feature in original_features:
        if transformed_name == f"num__{original_feature}" or transformed_name.startswith(f"cat__{original_feature}_"):
            return original_feature
    return transformed_name

feature_importance_by_original = feature_importance.assign(
    original_feature=feature_importance["feature"].map(lambda name: original_feature_name(name, X.columns))
).groupby("original_feature", as_index=False)["importance"].sum().sort_values("importance", ascending=False)
feature_importance_by_original["cumulative_importance"] = feature_importance_by_original["importance"].cumsum() / feature_importance_by_original["importance"].sum()

cutoff_position = np.flatnonzero(feature_importance_by_original["cumulative_importance"].to_numpy() >= 0.90)[0]
reduced_features = feature_importance_by_original.iloc[: cutoff_position + 1]["original_feature"].tolist()
print(f"Selected reduced feature count: {len(reduced_features)}")
display(pd.DataFrame({"selected_feature": reduced_features}))
reduced_numeric_features = [feature for feature in reduced_features if feature in numeric_features]
reduced_categorical_features = [feature for feature in reduced_features if feature in categorical_features]

reduced_preprocessor_tree = ColumnTransformer(
    transformers=[
        ("num", numeric_tree, reduced_numeric_features),
        ("cat", categorical, reduced_categorical_features),
    ]
)

best_xgb_params = {key.replace("model__", ""): value for key, value in xgb_search.best_params_.items()}
reduced_xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", reduced_preprocessor_tree),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=1,
            **best_xgb_params,
        )),
    ]
)

reduced_cv_scores = cross_val_score(reduced_xgb_pipeline, X_train, y_train, scoring="roc_auc", cv=cv, n_jobs=1)
reduced_xgb_pipeline.fit(X_train, y_train)
reduced_validation_metrics = evaluate_classifier("XGBoost reduced", reduced_xgb_pipeline, X_val, y_val)

full_xgb_metrics = next(result for result in results if result["model"] == "XGBoost")
feature_set_comparison = pd.DataFrame([
    {"model": "XGBoost full", "n_features": len(X.columns), "cv_roc_auc": xgb_search.best_score_, **{key: value for key, value in full_xgb_metrics.items() if key != "model"}},
    {"model": "XGBoost reduced", "n_features": len(reduced_features), "cv_roc_auc": reduced_cv_scores.mean(), **{key: value for key, value in reduced_validation_metrics.items() if key != "model"}},
])
display(feature_set_comparison.style.format({
    "cv_roc_auc": "{:.3f}",
    "roc_auc": "{:.3f}",
    "precision": "{:.3f}",
    "recall": "{:.3f}",
    "f1": "{:.3f}",
    "accuracy": "{:.3f}",
}))

The full feature set clearly outperforms the reduced version on validation ROC AUC (0.917 versus 0.858) and on the threshold metrics. We therefore retain the full feature set and continue with the engineered-feature comparison as a separate experiment.

## 12. XGBoost with engineered features

We test whether simple booking-time features improve the full XGBoost model. The engineered variables summarize stay length, party size, family status, previous booking history, and special requests. They use only fields available in the modeling dataset and do not use the target or post-booking outcome fields.

In [ ]:
def add_engineered_features(data):
    engineered = data.copy()
    engineered["total_nights"] = engineered["stays_in_weekend_nights"] + engineered["stays_in_week_nights"]
    engineered["total_guests"] = engineered["adults"] + engineered["children"].fillna(0) + engineered["babies"]
    engineered["is_family"] = ((engineered["children"].fillna(0) > 0) | (engineered["babies"] > 0)).astype(int)
    engineered["has_previous_cancellation"] = (engineered["previous_cancellations"] > 0).astype(int)
    engineered["has_previous_booking"] = (engineered["previous_bookings_not_canceled"] > 0).astype(int)
    engineered["has_special_requests"] = (engineered["total_of_special_requests"] > 0).astype(int)
    return engineered

X_train_engineered = add_engineered_features(X_train)
X_val_engineered = add_engineered_features(X_val)
engineered_numeric_features = X_train_engineered.select_dtypes(include=["number"]).columns.tolist()
engineered_categorical_features = X_train_engineered.select_dtypes(include=["str"]).columns.tolist()

engineered_preprocessor_tree = ColumnTransformer(
    transformers=[
        ("num", numeric_tree, engineered_numeric_features),
        ("cat", categorical, engineered_categorical_features),
    ]
)
engineered_xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", engineered_preprocessor_tree),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=1,
            **best_xgb_params,
        )),
    ]
)

engineered_cv_scores = cross_val_score(engineered_xgb_pipeline, X_train_engineered, y_train, scoring="roc_auc", cv=cv, n_jobs=1)
engineered_xgb_pipeline.fit(X_train_engineered, y_train)
engineered_validation_metrics = evaluate_classifier("XGBoost engineered", engineered_xgb_pipeline, X_val_engineered, y_val)

feature_set_comparison_all = pd.concat([
    feature_set_comparison,
    pd.DataFrame([{"model": "XGBoost engineered", "n_features": len(X_train_engineered.columns), "cv_roc_auc": engineered_cv_scores.mean(), **{key: value for key, value in engineered_validation_metrics.items() if key != "model"}}]),
], ignore_index=True)
display(feature_set_comparison_all.style.format({
    "cv_roc_auc": "{:.3f}",
    "roc_auc": "{:.3f}",
    "precision": "{:.3f}",
    "recall": "{:.3f}",
    "f1": "{:.3f}",
    "accuracy": "{:.3f}",
}))

The engineered version slightly improves validation ROC AUC in the current ablation, from 0.917 to 0.918, but this difference is marginal and the engineered version was not independently tuned. We therefore retain the full XGBoost pipeline as the final workflow. The reduced feature set is not selected because it performs materially worse.

## 13. XGBoost threshold analysis

We selected the full-feature XGBoost workflow, but the classification threshold controls the trade-off between identifying cancellations and generating false positives. We evaluate several thresholds on the validation probabilities to make this trade-off explicit before building the training and prediction scripts.

In [ ]:
selected_pipeline = xgb_search.best_estimator_
xgb_validation_probabilities = selected_pipeline.predict_proba(X_val)[:, 1]
thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]

threshold_results = []
for threshold in thresholds:
    threshold_predictions = (xgb_validation_probabilities >= threshold).astype(int)
    threshold_precision = precision_score(y_val, threshold_predictions, zero_division=0)
    threshold_recall = recall_score(y_val, threshold_predictions, zero_division=0)
    threshold_f1 = (
        2 * threshold_precision * threshold_recall / (threshold_precision + threshold_recall)
        if (threshold_precision + threshold_recall)
        else 0.0
    )
    threshold_results.append({
        "threshold": threshold,
        "precision": threshold_precision,
        "recall": threshold_recall,
        "f1": threshold_f1,
        "accuracy": accuracy_score(y_val, threshold_predictions),
    })

threshold_results = pd.DataFrame(threshold_results)
display(threshold_results.style.format({
    "threshold": "{:.2f}",
    "precision": "{:.3f}",
    "recall": "{:.3f}",
    "f1": "{:.3f}",
    "accuracy": "{:.3f}",
}))

Lowering the threshold increases cancellation recall at the cost of precision, while raising it has the opposite effect. We select threshold 0.40 because it provides the best validation balance under our decision criteria. We prioritize F1 because it summarizes the harmony between precision and recall, while we give additional weight to recall because the business assumption is that missing a real cancellation is more costly than contacting a guest unnecessarily.

A lower threshold helps identify more potential cancellations early enough for the hotel to confirm the booking, adjust inventory, offer the room again, or contact guests on a waiting list. The dataset does not include monetary costs, so this decision is based on the documented operational assumption rather than a cost-optimized threshold. The training and prediction workflow will use the full-feature XGBoost pipeline with the selected threshold.

In [ ]:
selected_threshold = 0.40

## 14. Final holdout evaluation

After selecting the XGBoost model and threshold on the training and validation data, we refit the full-feature pipeline on the combined train and validation partitions. We then evaluate it once on the untouched test set.

In [ ]:
from sklearn.base import clone

final_pipeline = clone(xgb_search.best_estimator_)
final_pipeline.fit(X_train_val, y_train_val)
final_train_metrics = evaluate_classifier(
    "Final XGBoost pipeline — train+validation",
    final_pipeline,
    X_train_val,
    y_train_val,
    threshold=selected_threshold,
)
final_test_metrics = evaluate_classifier(
    "Final XGBoost pipeline",
    final_pipeline,
    X_test,
    y_test,
    threshold=selected_threshold,
)
final_train_metrics["dataset"] = "Train + validation"
final_test_metrics["dataset"] = "Test"
cv_metrics = {
    "dataset": "Cross-validation (training partition)",
    "model": "XGBoost tuning model",
    "roc_auc": xgb_search.best_score_,
    "precision": np.nan,
    "recall": np.nan,
    "f1": np.nan,
    "accuracy": np.nan,
}
final_metrics_comparison = pd.DataFrame([cv_metrics, final_train_metrics, final_test_metrics]).set_index("dataset")
display(final_metrics_comparison.style.format({
    "roc_auc": "{:.3f}",
    "precision": "{:.3f}",
    "recall": "{:.3f}",
    "f1": "{:.3f}",
    "accuracy": "{:.3f}",
}))

The final metrics table compares the final full-feature XGBoost pipeline across cross-validation, train+validation, and test. Cross-validation on the training partition reports ROC AUC only because precision, recall, F1, and accuracy depend on a classification threshold and were not aggregated during CV. Training metrics are expected to be optimistic because the final pipeline was fitted on those observations. On train+validation, the final pipeline achieves ROC AUC 0.941, precision 0.740, recall 0.815, F1 0.776, and accuracy 0.870. On the untouched test set, it achieves ROC AUC 0.918, precision 0.698, recall 0.767, F1 0.731, and accuracy 0.845. The ROC AUC gap between cross-validation (0.916) and test (0.918) is very small, which supports good ranking generalization; the larger train/test gaps in threshold metrics are consistent with the expected optimism of in-sample evaluation. These test metrics are used in the project summary and deployment documentation.

## 15. Next modeling steps

The full-feature XGBoost pipeline and selected threshold are finalized using the validation comparison and then evaluated on the untouched test set. We can now extract the selected feature policy and model configuration into `train.py` and `predict.py`, preserving the preprocessing pipeline and the tuned hyperparameters. The prediction workflow will use the selected validation threshold together with the XGBoost pipeline.

Before deployment, we should save the fitted pipeline with a documented feature schema and preserve the selected feature set, threshold, and business assumption in the deployment documentation. A temporal validation experiment can be added later if we want to estimate performance on future arrival periods rather than a random stratified holdout.